# InHire Jobs Scraper
Radar automático de vagas de Data / BI / Analytics em empresas que usam InHire.

In [ ]:
import requests
import pandas as pd
import unicodedata
import time

## Função para normalizar texto (remove acentos e ignora case)

In [ ]:
def normalize(text):
    if not text:
        return ""
    text = text.lower()
    text = unicodedata.normalize('NFD', text)
    text = text.encode('ascii', 'ignore').decode("utf-8")
    return text

## Palavras-chave de vagas de dados

In [ ]:
KEYWORDS = [
"data",
"dados",
"analista de dados",
"data analyst",
"analytics",
"bi",
"business intelligence",
"cientista de dados",
"data scientist",
"engenheiro de dados",
"data engineer"
]

## Lista de empresas que usam InHire

In [ ]:
COMPANIES = [
"agenciacriativa","agrosearch","alice","alun","amcom","appcargo",
"auvotecnologia","bankme","betaonline","brasilparalelo","ceisc",
"cerc","cielo","coopersystem","cora","crown","db1","deloitte",
"digix","dito","docket","evolux","fitenergia","flash","flutterbrazil",
"fretadao","gex","grupoescalar","gx2","holu","infotecbrasil",
"kanastra","kpmg","linkapi","livemode","magalu","magazord",
"milvus","mjv","nomadglobal","olist","onfly","openlabs",
"orizon","pagar-me","paketa","paytrack","pier","pontte",
"premiersoft","qive","radix","rpo-recargapay","residclub",
"samsungsds","sanar","sharepeoplehub","shareprime","supertex",
"sylvamo","sympla","talentt","talentx","tripla","unimar",
"upflux","v360","v4company","vitru","warren","xerpa","zig",

"contabilizei","kiwify","loft","nubank","creditas","ifood","stone","loggi"
]

## Buscar vagas via API pública da InHire

In [ ]:
def fetch_jobs(company):

    jobs = []

    url = "https://api.inhire.app/job-posts/public"

    headers = {
        "X-Tenant": company,
        "User-Agent": "Mozilla/5.0"
    }

    params = {
        "page": 0,
        "size": 100
    }

    try:

        r = requests.get(url, headers=headers, params=params, timeout=10)

        if r.status_code != 200:
            return jobs

        data = r.json()

        if "content" not in data:
            return jobs

        for job in data["content"]:

            title = job.get("title","")
            slug = job.get("slug","")
            job_id = job.get("id","")

            link = f"https://{company}.inhire.app/vagas/{job_id}/{slug}"

            jobs.append({
                "empresa": company,
                "titulo": title,
                "link": link
            })

    except Exception as e:
        pass

    return jobs

## Função para identificar vagas de dados

In [ ]:
def is_data_job(title):

    title_norm = normalize(title)

    for k in KEYWORDS:

        if normalize(k) in title_norm:

            return True

    return False

## Scanner de vagas

In [ ]:
all_jobs = []

for company in COMPANIES:

    print("Escaneando:", company)

    jobs = fetch_jobs(company)

    all_jobs.extend(jobs)

    time.sleep(0.3)

## Filtrar vagas de dados

In [ ]:
data_jobs = []

for job in all_jobs:

    if is_data_job(job["titulo"]):

        data_jobs.append(job)

## Estatísticas

In [ ]:
print("Total vagas encontradas:", len(all_jobs))
print("Vagas de dados encontradas:", len(data_jobs))

## Salvar resultado

In [ ]:
df = pd.DataFrame(data_jobs)

df.to_csv("vagas_data_analyst.csv", index=False)

print("Arquivo salvo: vagas_data_analyst.csv")

df